In [ ]:
# ================================
# CELLULE 3: CORRECTION DES PROBLÈMES D'IMPORTATION
# ================================

print("="*60)
print("CORRECTION DES PROBLÈMES DE COMPATIBILITÉ")
print("="*60)

# Résoudre les problèmes de protobuf
print("1. Correction des problèmes de protobuf...")
try:
    import pkg_resources
    import types
    
    # Vérifier si protobuf est à jour
    !pip install --upgrade "protobuf<4.0.0" --quiet
    print("✓ protobuf mis à jour")
except Exception as e:
    print(f"⚠ Problème avec protobuf: {e}, mais continuons...")

# IMPORTATION de TensorFlow d'abord
import tensorflow as tf

print("2. Vérification de l'environnement Kaggle...")
print(f"✓ TensorFlow version: {tf.__version__}")
print(f"✓ GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

if tf.config.list_physical_devices('GPU'):
    print("✅ GPU détecté et prêt à l'emploi!")
    # Configurer la mémoire GPU si disponible
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(f"Erreur configuration GPU: {e}")
else:
    print("❌ Pas de GPU disponible, l'entraînement sera lent")

print("3. Importation des autres bibliothèques...")
# Importer les autres bibliothèques
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
import shutil
import glob
import time
import json
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Vérifier si rarfile est disponible pour les archives RAR
try:
    import rarfile
    print("✓ rarfile disponible")
except:
    print("⚠ rarfile non disponible (installer avec: pip install rarfile)")

print("✓ Toutes les bibliothèques sont prêtes")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")

In [ ]:
from IPython.display import FileLink, display
import os

In [ ]:
# ================================
# CELLULE 4: VÉRIFICATION DU DATASET
# ================================

print("="*60)
print("VÉRIFICATION DU DATASET")
print("="*60)

# Votre dataset a été téléversé sous le nom 'datasetcnn'
# Vérifions sa structure

dataset_path = '/kaggle/input/datasetcnn/data'
print(f"Chemin du dataset: {dataset_path}")

if os.path.exists(dataset_path):
    print("✅ Dataset trouvé!")
    print("\nStructure du dataset:")
    
    # Afficher la structure
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # Afficher seulement les 5 premiers fichiers
            print(f"{subindent}{file}")
        if len(files) > 5:
            print(f"{subindent}... et {len(files) - 5} autres fichiers")
    
    # Chercher spécifiquement les dossiers train, valid, test
    print("\n🔍 Recherche des dossiers d'entraînement...")
    
    # Fonction pour trouver les dossiers
    def find_data_folders(base_path):
        folders = {}
        for item in os.listdir(base_path):
            item_path = os.path.join(base_path, item)
            if os.path.isdir(item_path):
                # Vérifier si ce dossier contient des sous-dossiers (classes)
                subdirs = [d for d in os.listdir(item_path) if os.path.isdir(os.path.join(item_path, d))]
                if len(subdirs) >= 2:  # Au moins 2 classes
                    folders[item] = item_path
        
        return folders
    
    # Chercher dans le dossier principal et un niveau plus bas
    all_folders = {}
    
    # Niveau 1
    folders_level1 = find_data_folders(dataset_path)
    all_folders.update(folders_level1)
    
    # Niveau 2 (si le dataset est dans un sous-dossier 'data')
    for subfolder in os.listdir(dataset_path):
        subfolder_path = os.path.join(dataset_path, subfolder)
        if os.path.isdir(subfolder_path):
            folders_level2 = find_data_folders(subfolder_path)
            for key, value in folders_level2.items():
                all_folders[f"{subfolder}/{key}"] = value
    
    print("\n📁 Dossiers de données trouvés:")
    for name, path in all_folders.items():
        # Compter les images
        total_images = 0
        class_counts = {}
        for class_name in os.listdir(path):
            class_path = os.path.join(path, class_name)
            if os.path.isdir(class_path):
                images = [f for f in os.listdir(class_path) 
                         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
                class_counts[class_name] = len(images)
                total_images += len(images)
        
        print(f"• {name}: {total_images} images totales")
        for class_name, count in class_counts.items():
            print(f"    - {class_name}: {count} images")
    
else:
    print("❌ Dataset non trouvé!")
    print("\nVérifiez que:")
    print("1. Vous avez bien téléversé votre fichier .rar")
    print("2. Vous avez sélectionné 'datasetcnn' comme dataset")
    print("3. Le dataset est bien dans /kaggle/input/")
    
    # Lister les datasets disponibles
    print("\n📦 Datasets disponibles dans /kaggle/input/:")
    available_datasets = os.listdir('/kaggle/input')
    for ds in available_datasets:
        print(f"  - {ds}")

In [ ]:
# ================================
# CELLULE 5: DÉFINITION DES CHEMINS FINAUX
# ================================

print("="*60)
print("CONFIGURATION DES CHEMINS")
print("="*60)

# Basé sur votre structure (data.rar contenant data/train, data/valid, data/test)
# Le dataset est déjà extrait par Kaggle

# Essayer plusieurs structures possibles
possible_paths = [
    '/kaggle/input/datasetcnn/data',  # Votre structure probable
    '/kaggle/input/datasetcnn/datasetcnn/data',
    '/kaggle/input/datasetcnn',
]

for base_path in possible_paths:
    if os.path.exists(base_path):
        print(f"✅ Base path trouvé: {base_path}")
        
        # Chercher les dossiers train, valid, test
        train_candidates = ['train', 'training', 'train_data', 'train_set']
        valid_candidates = ['valid', 'validation', 'val', 'val_data', 'val_set']
        test_candidates = ['test', 'testing', 'test_data', 'test_set']
        
        train_dir = None
        valid_dir = None
        test_dir = None
        
        # Chercher train
        for candidate in train_candidates:
            candidate_path = os.path.join(base_path, candidate)
            if os.path.exists(candidate_path):
                train_dir = candidate_path
                print(f"  ✓ Train: {train_dir}")
                break
        
        # Chercher validation
        for candidate in valid_candidates:
            candidate_path = os.path.join(base_path, candidate)
            if os.path.exists(candidate_path):
                valid_dir = candidate_path
                print(f"  ✓ Valid: {valid_dir}")
                break
        
        # Chercher test
        for candidate in test_candidates:
            candidate_path = os.path.join(base_path, candidate)
            if os.path.exists(candidate_path):
                test_dir = candidate_path
                print(f"  ✓ Test: {test_dir}")
                break
        
        # Si train trouvé, utiliser celui-ci
        if train_dir:
            break

# Si non trouvés, chercher récursivement
if not train_dir:
    print("🔍 Recherche récursive des dossiers...")
    for root, dirs, files in os.walk('/kaggle/input/datasetcnn'):
        for dir_name in dirs:
            dir_path = os.path.join(root, dir_name)
            # Vérifier si ce dossier contient des sous-dossiers (classes)
            subdirs = [d for d in os.listdir(dir_path) if os.path.isdir(os.path.join(dir_path, d))]
            if len(subdirs) >= 2:  # Au moins 2 classes
                if 'train' in dir_name.lower():
                    train_dir = dir_path
                elif 'val' in dir_name.lower() or 'valid' in dir_name.lower():
                    valid_dir = dir_path
                elif 'test' in dir_name.lower():
                    test_dir = dir_path

# Afficher les chemins finaux
print(f"\n📍 CHEMINS FINAUX:")
print(f"Train: {train_dir if train_dir else 'NON TROUVÉ'}")
print(f"Valid: {valid_dir if valid_dir else 'NON TROUVÉ'}")
print(f"Test: {test_dir if test_dir else 'NON TROUVÉ'}")

# Vérifier l'existence et compter les images
print(f"\n📊 VÉRIFICATION DES DONNÉES:")

for name, path in [("Train", train_dir), ("Valid", valid_dir), ("Test", test_dir)]:
    if path and os.path.exists(path):
        # Compter les images par classe
        class_counts = {}
        total_images = 0
        
        for class_name in os.listdir(path):
            class_path = os.path.join(path, class_name)
            if os.path.isdir(class_path):
                images = [f for f in os.listdir(class_path) 
                         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
                count = len(images)
                class_counts[class_name] = count
                total_images += count
        
        print(f"✅ {name}: {total_images} images")
        for class_name, count in class_counts.items():
            print(f"   - {class_name}: {count} images")
    else:
        print(f"❌ {name}: Dossier non trouvé")

# Si train_dir n'est pas trouvé, demander manuellement
if not train_dir:
    print("\n❌ Impossible de trouver automatiquement les dossiers.")
    print("Veuillez spécifier manuellement les chemins:")
    
    # Lister tous les dossiers possibles
    print("\nDossiers disponibles dans /kaggle/input/datasetcnn:")
    for root, dirs, files in os.walk('/kaggle/input/datasetcnn'):
        level = root.replace('/kaggle/input/datasetcnn', '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root) if root != '/kaggle/input/datasetcnn' else 'datasetcnn'}/")
        subindent = ' ' * 2 * (level + 1)
        for dir_name in dirs:
            print(f"{subindent}{dir_name}/")
    
    train_dir = input("\nChemin complet du dossier train: ").strip()
    valid_dir = input("Chemin complet du dossier valid: ").strip()
    test_dir = input("Chemin complet du dossier test: ").strip()

In [ ]:
# ================================
# CELLULE 6: PARAMÈTRES DU MODÈLE
# ================================

print("="*60)
print("CONFIGURATION DES PARAMÈTRES")
print("="*60)

# Paramètres d'image
IMG_SIZE = 224  # Standard pour la plupart des modèles pré-entraînés
BATCH_SIZE = 32  # Peut être ajusté selon la mémoire GPU

# Paramètres d'entraînement
EPOCHS_PHASE1 = 10  # Phase 1 avec modèle gelé (réduit pour tests rapides)
EPOCHS_PHASE2 = 5   # Phase 2 fine-tuning (réduit pour tests rapides)

# Taux d'apprentissage
LEARNING_RATE_PHASE1 = 1e-3
LEARNING_RATE_PHASE2 = 1e-5

print("⚙️ PARAMÈTRES:")
print(f"• Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"• Batch Size: {BATCH_SIZE}")
print(f"• Epochs Phase 1: {EPOCHS_PHASE1}")
print(f"• Epochs Phase 2: {EPOCHS_PHASE2}")
print(f"• Learning Rate Phase 1: {LEARNING_RATE_PHASE1}")
print(f"• Learning Rate Phase 2: {LEARNING_RATE_PHASE2}")

print(f"\n🎮 ENVIRONNEMENT KAGGLE:")
# Vérifier la mémoire disponible
import psutil
memory = psutil.virtual_memory()
print(f"• RAM totale: {memory.total / (1024**3):.1f} GB")
print(f"• RAM disponible: {memory.available / (1024**3):.1f} GB")

# Vérifier le GPU
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"• GPU: {len(gpu_devices)} disponible(s)")
    # Afficher des infos sur le GPU
    try:
        gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
        print(f"• GPU Info: {gpu_info[0] if gpu_info else 'Inconnu'}")
    except:
        print("• GPU Info: Disponible mais informations non accessibles")
else:
    print("• GPU: Aucun disponible (entraînement sur CPU)")

In [ ]:
# ================================
# CELLULE 7: PRÉPARATION DES DONNÉES
# ================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("="*60)
print("PRÉPARATION DES DONNÉES")
print("="*60)

def prepare_data():
    """Prépare les générateurs de données"""
    
    print("🔄 Préparation des générateurs d'images...")
    
    # Augmentation de données pour l'entraînement
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    
    # Pas d'augmentation pour validation/test
    val_test_datagen = ImageDataGenerator(rescale=1./255)
    
    print("📥 Chargement des données...")
    
    try:
        # Train generator
        train_generator = train_datagen.flow_from_directory(
            train_dir,
            target_size=(IMG_SIZE, IMG_SIZE),
            batch_size=BATCH_SIZE,
            class_mode='binary',
            shuffle=True,
            seed=42
        )
        
        # Validation generator
        val_generator = val_test_datagen.flow_from_directory(
            valid_dir if valid_dir else train_dir,  # Si pas de valid, utiliser train
            target_size=(IMG_SIZE, IMG_SIZE),
            batch_size=BATCH_SIZE,
            class_mode='binary',
            shuffle=False
        )
        
        # Test generator
        test_generator = val_test_datagen.flow_from_directory(
            test_dir if test_dir else valid_dir if valid_dir else train_dir,
            target_size=(IMG_SIZE, IMG_SIZE),
            batch_size=BATCH_SIZE,
            class_mode='binary',
            shuffle=False
        )
        
        print(f"\n✅ DONNÉES CHARGÉES AVEC SUCCÈS!")
        print(f"• Classes: {train_generator.class_indices}")
        print(f"• Train: {train_generator.samples} images")
        print(f"• Validation: {val_generator.samples} images")
        print(f"• Test: {test_generator.samples} images")
        
        # Afficher un échantillon
        print("\n🔍 Aperçu d'un batch:")
        batch_x, batch_y = next(train_generator)
        print(f"  Shape des images: {batch_x.shape}")
        print(f"  Shape des labels: {batch_y.shape}")
        print(f"  5 premiers labels: {batch_y[:5].flatten()}")
        
        return train_generator, val_generator, test_generator
        
    except Exception as e:
        print(f"❌ ERREUR lors du chargement des données: {e}")
        print("\nVérifiez que:")
        print("1. Les chemins sont corrects")
        print("2. Les dossiers contiennent des sous-dossiers de classes")
        print("3. Il y a des images dans chaque sous-dossier")
        raise e

# Préparer les données
train_gen, val_gen, test_gen = prepare_data()

In [ ]:
# ================================
# CELLULE 8: DÉFINITION DES MODÈLES
# ================================

print("="*60)
print("DÉFINITION DES ARCHITECTURES")
print("="*60)

def create_efficientnet():
    """Crée un modèle EfficientNetB0"""
    print("🏗️  Construction d'EfficientNetB0...")
    
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling='avg'
    )
    
    model = models.Sequential([
        base_model,
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    
    print(f"✓ EfficientNetB0 créé ({model.count_params():,} paramètres)")
    return model, base_model

def create_resnet():
    """Crée un modèle ResNet50V2"""
    print("🏗️  Construction de ResNet50V2...")
    
    base_model = ResNet50V2(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling='avg'
    )
    
    model = models.Sequential([
        base_model,
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    
    print(f"✓ ResNet50V2 créé ({model.count_params():,} paramètres)")
    return model, base_model

# Vous pouvez ajouter d'autres modèles si nécessaire
print("✅ Fonctions de modèles définies")

In [ ]:
# ================================
# CELLULE 9: FONCTION D'ENTRAÎNEMENT
# ================================

print("="*60)
print("FONCTION D'ENTRAÎNEMENT COMPLÈTE")
print("="*60)

def train_model(model_type='efficientnet'):
    """Entraîne un modèle complet en deux phases"""
    
    print(f"\n{'🚀'*30}")
    print(f"ENTRAÎNEMENT DU MODÈLE: {model_type.upper()}")
    print(f"{'🚀'*30}")
    
    # 1. Créer le modèle
    if model_type == 'efficientnet':
        model, base_model = create_efficientnet()
    elif model_type == 'resnet':
        model, base_model = create_resnet()
    else:
        raise ValueError(f"Modèle non supporté: {model_type}")
    
    # 2. Phase 1: Modèle gelé
    print(f"\n📚 PHASE 1: Entraînement avec modèle gelé")
    print("-" * 50)
    
    # Geler le modèle de base
    base_model.trainable = False
    
    # Compiler le modèle
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_PHASE1),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.AUC(name='auc')
        ]
    )
    
    # Calculer les steps
    train_steps = max(1, train_gen.samples // BATCH_SIZE)
    val_steps = max(1, val_gen.samples // BATCH_SIZE)
    
    print(f"⚙️  Paramètres Phase 1:")
    print(f"  Steps par epoch: {train_steps}")
    print(f"  Validation steps: {val_steps}")
    print(f"  Learning rate: {LEARNING_RATE_PHASE1}")
    
    # Callbacks Phase 1
    callbacks_phase1 = [
        EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            f'/kaggle/working/best_{model_type}_phase1.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]
    
    # Entraînement Phase 1
    print("\n▶️ Début de l'entraînement Phase 1...")
    history1 = model.fit(
        train_gen,
        steps_per_epoch=train_steps,
        epochs=EPOCHS_PHASE1,
        validation_data=val_gen,
        validation_steps=val_steps,
        callbacks=callbacks_phase1,
        verbose=1
    )
    
    # Télécharger immédiatement le modèle phase 1
    print("\n⬇️  Téléchargement du modèle Phase 1...")
    display(FileLink(f'best_{model_type}_phase1.h5'))
    
    # 3. Phase 2: Fine-tuning
    print(f"\n🎯 PHASE 2: Fine-tuning")
    print("-" * 50)
    
    # Dégeler le modèle de base
    base_model.trainable = True
    
    # Dégeler seulement les dernières couches
    unfrozen_layers = 100
    for layer in base_model.layers[:-unfrozen_layers]:
        layer.trainable = False
    
    print(f"✓ {unfrozen_layers} couches dégelées sur {len(base_model.layers)}")
    
    # Recompiler avec learning rate plus faible
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_PHASE2),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall', 'auc']
    )
    
    print(f"⚙️  Paramètres Phase 2:")
    print(f"  Steps par epoch: {train_steps}")
    print(f"  Validation steps: {val_steps}")
    print(f"  Learning rate: {LEARNING_RATE_PHASE2}")
    
    # Callbacks Phase 2
    callbacks_phase2 = [
        EarlyStopping(
            monitor='val_loss',
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=4,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            f'/kaggle/working/best_{model_type}_phase2.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]
    
    # Entraînement Phase 2
    print("\n▶️ Début de l'entraînement Phase 2...")
    history2 = model.fit(
        train_gen,
        steps_per_epoch=train_steps,
        epochs=EPOCHS_PHASE2,
        validation_data=val_gen,
        validation_steps=val_steps,
        callbacks=callbacks_phase2,
        verbose=1
    )
    
    # Télécharger immédiatement le modèle phase 2
    print("\n⬇️  Téléchargement du modèle Phase 2...")
    display(FileLink(f'best_{model_type}_phase2.h5'))
    
    # 4. Évaluation finale
    print(f"\n📊 ÉVALUATION FINALE")
    print("-" * 50)
    
    # Sauvegarder le modèle final
    final_model_path = f'model_final_{model_type}.h5'
    model.save(f'/kaggle/working/{final_model_path}')
    print(f"💾 Modèle final sauvegardé: {final_model_path}")
    
    # Télécharger immédiatement le modèle final
    print("⬇️  Téléchargement du modèle final...")
    display(FileLink(final_model_path))
    
    # Évaluation sur le test set
    test_gen.reset()
    test_steps = max(1, test_gen.samples // BATCH_SIZE)
    
    print("\n🧪 Évaluation sur le dataset de test...")
    test_results = model.evaluate(test_gen, steps=test_steps, verbose=1)
    
    # Collecter les prédictions détaillées
    test_gen.reset()
    y_true = []
    y_pred_proba = []
    
    for i in range(test_steps + 1):
        try:
            images, labels = next(test_gen)
            preds = model.predict(images, verbose=0)
            y_true.extend(labels)
            y_pred_proba.extend(preds.flatten())
        except:
            break
    
    # Convertir en classes
    y_pred = [1 if p > 0.5 else 0 for p in y_pred_proba]
    
    # Calculer les métriques
    cm = confusion_matrix(y_true, y_pred)
    cr = classification_report(y_true, y_pred, target_names=['Real', 'Fake'], output_dict=True)
    auc_score = roc_auc_score(y_true, y_pred_proba)
    
    # Afficher les résultats
    print(f"\n📈 RÉSULTATS FINAUX - {model_type.upper()}:")
    print(f"• Accuracy: {test_results[1]:.4f}")
    print(f"• Precision: {test_results[2]:.4f}")
    print(f"• Recall: {test_results[3]:.4f}")
    print(f"• AUC: {test_results[4]:.4f}")
    
    print(f"\n📋 MATRICE DE CONFUSION:")
    print(cm)
    
    print(f"\n📄 RAPPORT DE CLASSIFICATION:")
    print(classification_report(y_true, y_pred, target_names=['Real', 'Fake']))
    
    # Préparer les résultats
    results = {
        'model': model,
        'history1': history1.history,
        'history2': history2.history,
        'test_results': test_results,
        'confusion_matrix': cm,
        'classification_report': cr,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'auc_score': auc_score
    }
    
    return results

In [ ]:
# ================================
# CELLULE 11: ENTRAÎNEMENT RESNET50V2
# ================================

from tensorflow.keras. applications import EfficientNetB0, ResNet50, VGG16
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras. callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import json
import time

print("="*60)
print("DÉBUT DE L'ENTRAÎNEMENT EFFICIENTNETB0")
print("="*60)

import time

# Démarrage du timer
start_time = time.time()
from tensorflow.keras. applications import  ResNet50V2

print("="*60)
print("DÉBUT DE L'ENTRAÎNEMENT RESNET50V2")
print("="*60)

# Vérifier si EfficientNet a réussi
if 'eff_results' not in locals():
    print("⚠️  EfficientNet n'a pas été entraîné ou a échoué.")
    print("Continuer avec ResNet malgré tout...")

# Démarrage du timer
start_time = time.time()

try:
    print("🔥 Initialisation de l'entraînement de ResNet50V2...")
    
    # Entraîner le modèle
    resnet_results = train_model('resnet')
    
    # Calcul du temps
    training_time = time.time() - start_time
    hours = int(training_time // 3600)
    minutes = int((training_time % 3600) // 60)
    seconds = int(training_time % 60)
    
    print(f"\n{'✅'*20}")
    print("ENTRAÎNEMENT RESNET50V2 TERMINÉ!")
    print(f"{'✅'*20}")
    
    print(f"\n⏱️  Temps total d'entraînement: {hours}h {minutes}m {seconds}s")
    
    # Résumé des performances
    print(f"\n🏆 PERFORMANCES RESNET50V2:")
    print(f"• Test Accuracy: {resnet_results['test_results'][1]:.4f}")
    print(f"• Test Precision: {resnet_results['test_results'][2]:.4f}")
    print(f"• Test Recall: {resnet_results['test_results'][3]:.4f}")
    print(f"• Test AUC: {resnet_results['test_results'][4]:.4f}")
    
    # Sauvegarder les résultats dans un fichier JSON
    results_to_save = {
        'model_name': 'ResNet50V2',
        'test_accuracy': float(resnet_results['test_results'][1]),
        'test_precision': float(resnet_results['test_results'][2]),
        'test_recall': float(resnet_results['test_results'][3]),
        'test_auc': float(resnet_results['test_results'][4]),
        'training_time_seconds': training_time,
        'training_time_human': f"{hours}h {minutes}m {seconds}s",
        'confusion_matrix': resnet_results['confusion_matrix'].tolist(),
        'classification_report': resnet_results['classification_report']
    }
    
    with open('/kaggle/working/resnet_results.json', 'w') as f:
        json.dump(results_to_save, f, indent=4)
    
    print("💾 Résultats sauvegardés dans /kaggle/working/resnet_results.json")
    
    # Visualisation de l'apprentissage
    print(f"\n📈 VISUALISATION DE L'APPRENTISSAGE")
    
    # Combiner les historiques
    combined_history = {}
    for key in resnet_results['history1'].keys():
        combined_history[key] = resnet_results['history1'][key] + resnet_results['history2'].get(key, [])
    
    # Créer les graphiques
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 1. Accuracy
    axes[0, 0].plot(combined_history['accuracy'], label='Train')
    axes[0, 0].plot(combined_history['val_accuracy'], label='Validation')
    axes[0, 0].axvline(x=EPOCHS_PHASE1, color='r', linestyle='--', label='Fine-tuning start')
    axes[0, 0].set_title('Accuracy')
    axes[0, 0].set_xlabel('Epochs')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # 2. Loss
    axes[0, 1].plot(combined_history['loss'], label='Train')
    axes[0, 1].plot(combined_history['val_loss'], label='Validation')
    axes[0, 1].axvline(x=EPOCHS_PHASE1, color='r', linestyle='--', label='Fine-tuning start')
    axes[0, 1].set_title('Loss')
    axes[0, 1].set_xlabel('Epochs')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # 3. Precision
    axes[0, 2].plot(combined_history['precision'], label='Train')
    axes[0, 2].plot(combined_history['val_precision'], label='Validation')
    axes[0, 2].axvline(x=EPOCHS_PHASE1, color='r', linestyle='--')
    axes[0, 2].set_title('Precision')
    axes[0, 2].set_xlabel('Epochs')
    axes[0, 2].set_ylabel('Precision')
    axes[0, 2].legend()
    axes[0, 2].grid(True)
    
    # 4. Recall
    axes[1, 0].plot(combined_history['recall'], label='Train')
    axes[1, 0].plot(combined_history['val_recall'], label='Validation')
    axes[1, 0].axvline(x=EPOCHS_PHASE1, color='r', linestyle='--')
    axes[1, 0].set_title('Recall')
    axes[1, 0].set_xlabel('Epochs')
    axes[1, 0].set_ylabel('Recall')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    # 5. AUC
    axes[1, 1].plot(combined_history['auc'], label='Train')
    axes[1, 1].plot(combined_history['val_auc'], label='Validation')
    axes[1, 1].axvline(x=EPOCHS_PHASE1, color='r', linestyle='--')
    axes[1, 1].set_title('AUC')
    axes[1, 1].set_xlabel('Epochs')
    axes[1, 1].set_ylabel('AUC')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    # 6. Matrice de confusion
    axes[1, 2].imshow(resnet_results['confusion_matrix'], cmap='Oranges')
    axes[1, 2].set_title('Matrice de Confusion (Test)')
    axes[1, 2].set_xticks([0, 1])
    axes[1, 2].set_yticks([0, 1])
    axes[1, 2].set_xticklabels(['Real', 'Fake'])
    axes[1, 2].set_yticklabels(['Real', 'Fake'])
    axes[1, 2].set_xlabel('Prédit')
    axes[1, 2].set_ylabel('Réel')
    
    for i in range(2):
        for j in range(2):
            axes[1, 2].text(j, i, str(resnet_results['confusion_matrix'][i, j]), 
                           ha='center', va='center', color='blue', fontsize=14, fontweight='bold')
    
    plt.suptitle('ResNet50V2 - Résultats d\'entraînement', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/kaggle/working/resnet_training.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("📊 Graphiques sauvegardés dans /kaggle/working/resnet_training.png")
    
except Exception as e:
    print(f"\n❌ ERREUR lors de l'entraînement de ResNet50V2: {e}")
    import traceback
    traceback.print_exc()
    print("\n⚠️  L'entraînement a échoué. Vérifiez les messages d'erreur ci-dessus.")